# H1 — Umidade do solo e necessidade de irrigação

Hipótese: menores níveis de umidade do solo estão associados a maior necessidade de irrigação. A irrigação é medida pelo volume de água, não pelo estado da válvula.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    ROOT = (Path.cwd() / '..').resolve()
sys.path.insert(0, str(ROOT))

from src.utils import extrair_eventos_irrigacao, plot_eventos_irrigacao, plot_proporcao_acionamento

In [2]:
df_h1 = pd.read_csv(ROOT / 'data' / 'processed' / 'dataset_m2_2025.csv', low_memory=False)
df_h1 = df_h1.loc[df_h1['line'].eq(5)].copy()
df_h1['safra'] = '2025'

In [3]:
# Eventos: guarda a amostra anterior a cada transição 0→1 e calcula o volume do evento.
timestamps_monitoramento = pd.to_datetime(df_h1['timestamp_10min'], errors='coerce', utc=True).dropna()
eventos_h1 = extrair_eventos_irrigacao(df_h1)
figura_scatter = plot_eventos_irrigacao(eventos_h1)
figura_proporcao, tabela_proporcao = plot_proporcao_acionamento(df_h1)
print(f'Início do monitoramento: {timestamps_monitoramento.min():%d/%m/%Y}')
print(f'Fim do monitoramento: {timestamps_monitoramento.max():%d/%m/%Y}')
print(f'Figuras salvas em: {figura_scatter} e {figura_proporcao}')
print(tabela_proporcao.to_string(index=False))

Início do monitoramento: 26/06/2025
Fim do monitoramento: 28/10/2025
Figuras salvas em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H1/H1_eventos_umidade_pre_abertura_vs_volume.png e /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H1/H1_proporcao_acionamento_por_faixa_de_umidade.png
faixa_umidade  total  aberturas  proporcao
  28.28–32.66   3490          0   0.000000
  32.66–36.08   3470          1   0.028818
  36.08–55.86   3479          0   0.000000


In [4]:
# Correlação por evento: umidade imediatamente antes da abertura × volume do evento.
rho = eventos_h1['soil_humidity_pre_abertura'].corr(
    eventos_h1['volume_evento'], method='spearman'
)
print(f'Correlação de Spearman por evento: {rho:.3f}')

Correlação de Spearman por evento: nan


In [5]:
# Tabela comparativa: linhas 4 e 5 em 2024; linhas 1 e 2 em 2025.
tabelas_eventos = []
for safra in ('2024', '2025'):
    dados_safra = pd.read_csv(ROOT / 'data' / 'processed' / f'dataset_m2_{safra}.csv', low_memory=False)
    linhas = [4] if safra == '2024' else [1]
    for linha in linhas:
        dados_linha = dados_safra.loc[dados_safra['line'].eq(linha)].copy()
        eventos_linha = extrair_eventos_irrigacao(dados_linha)
        eventos_linha.insert(0, 'line', linha)
        eventos_linha.insert(0, 'safra', safra)
        tabelas_eventos.append(eventos_linha)

tabela_linha = pd.concat(tabelas_eventos, ignore_index=True)
tabela_linha = tabela_linha[[
    'safra', 'line', 'evento', 'timestamp_pre_abertura',
    'soil_humidity_pre_abertura', 'timestamp_abertura',
    'soil_humidity_abertura', 'volume_evento'
]]
print(tabela_linha.to_string(index=False))

safra  line  evento    timestamp_pre_abertura  soil_humidity_pre_abertura        timestamp_abertura  soil_humidity_abertura  volume_evento
 2024     4       1 2024-07-26 06:50:00+00:00                      30.795 2024-07-26 07:00:00+00:00                  30.795         1607.0
 2024     4       2 2024-07-27 06:50:00+00:00                      28.855 2024-07-27 07:00:00+00:00                  28.855         1461.0
 2024     4       3 2024-07-29 06:50:00+00:00                      29.500 2024-07-29 07:00:00+00:00                  29.470         1725.0
 2024     4       4 2024-07-30 06:50:00+00:00                      31.450 2024-07-30 07:00:00+00:00                  31.455         1636.0
 2024     4       5 2024-08-02 07:00:00+00:00                      29.060 2024-08-02 07:10:00+00:00                  29.060         1579.0
 2024     4       6 2024-08-03 07:00:00+00:00                      29.410 2024-08-03 07:10:00+00:00                  29.435         1276.0
 2024     4       7 2024-08

In [7]:
# Boxplots por safra: volume equivalente a 100% da recomendação Irriframe.
# As linhas 5 (2024) e 2 (2025) receberam 30% da recomendação.
regime_irriframe = {
    ('2024', 4): 1.00,
    # ('2024', 5): 0.30,
    ('2025', 1): 1.00,
    # ('2025', 2): 0.30,
}
dados_boxplot = tabela_linha.copy()
dados_boxplot['fator_irriframe'] = [
    regime_irriframe[(str(safra), int(linha))]
    for safra, linha in zip(dados_boxplot['safra'], dados_boxplot['line'])
]
dados_boxplot['volume_equivalente_100'] = (
    dados_boxplot['volume_evento'] / dados_boxplot['fator_irriframe']
)

for safra in ('2024', '2025'):
    dados_ano = dados_boxplot.loc[dados_boxplot['safra'].eq(safra)].copy()
    dados_ano['faixa_umidade'] = pd.qcut(
        dados_ano['soil_humidity_pre_abertura'], q=3, duplicates='drop'
    )

    grupos = []
    rotulos = []
    for faixa in dados_ano['faixa_umidade'].cat.categories:
        valores = dados_ano.loc[
            dados_ano['faixa_umidade'].eq(faixa), 'volume_equivalente_100'
        ].dropna()
        if valores.empty:
            continue
        umidade = dados_ano.loc[
            dados_ano['faixa_umidade'].eq(faixa),
            'soil_humidity_pre_abertura',
        ]
        grupos.append(valores.to_numpy())
        rotulos.append(f'{umidade.min():.2f}–{umidade.max():.2f}')

    fig, ax = plt.subplots(figsize=(9, 6))
    boxplot = ax.boxplot(grupos, tick_labels=rotulos, patch_artist=True)
    for caixa, cor in zip(boxplot['boxes'], ('#c6dbef', '#6baed6', '#2171b5')):
        caixa.set_facecolor(cor)
    linha_safra = next(
        linha
        for (safra_dict, linha), fator in regime_irriframe.items()
        if safra_dict == safra and fator == 1.00
    )
    ax.set_title(f'Volume equivalente na linha {linha_safra} — {safra}')
    ax.set_xlabel('umidade antes da abertura (%RH)')
    ax.set_ylabel('Volume (m³)')
    ax.grid(axis='y', color='#c6dbef', alpha=0.45)
    fig.tight_layout()
    saida = ROOT / 'imgs' / 'H1' / f'H1_boxplot_volume_equivalente_100_{safra}.png'
    saida.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(saida, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Figura salva em: {saida}')

Figura salva em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H1/H1_boxplot_volume_equivalente_100_2024.png
Figura salva em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H1/H1_boxplot_volume_equivalente_100_2025.png


In [8]:
# Quantidade de amostras com a válvula ligada e desligada por linha e safra.
# Estados ausentes ou diferentes de 0/1 são ignorados.
contagens_valvula = []
for safra in ('2023', '2024', '2025'):
    dados_valvula = pd.read_csv(
        ROOT / 'data' / 'processed' / f'dataset_m2_{safra}.csv',
        usecols=['line', 'valve_state'],
        low_memory=False,
    )
    dados_valvula['valve_state'] = pd.to_numeric(
        dados_valvula['valve_state'], errors='coerce'
    )
    dados_valvula = dados_valvula.loc[
        dados_valvula['valve_state'].isin([0, 1])
    ].copy()
    dados_valvula['valvula_ligada'] = dados_valvula['valve_state'].eq(1)
    dados_valvula['valvula_desligada'] = dados_valvula['valve_state'].eq(0)

    contagem_safra = (
        dados_valvula.groupby('line', as_index=False)
        .agg(
            amostras_valvula_ligada=('valvula_ligada', 'sum'),
            amostras_valvula_desligada=('valvula_desligada', 'sum'),
        )
    )
    contagem_safra.insert(0, 'safra', safra)
    contagens_valvula.append(contagem_safra)

tabela_contagem_valvula = pd.concat(contagens_valvula, ignore_index=True)
print(tabela_contagem_valvula.to_string(index=False))

safra  line  amostras_valvula_ligada  amostras_valvula_desligada
 2023     1                     9962                         673
 2023     2                    10095                         675
 2023     3                    10082                           3
 2024     1                      135                        9115
 2024     2                      109                        9163
 2024     3                      229                        9021
 2024     4                      169                        9041
 2024     5                       53                        9183
 2025     1                       80                       16971
 2025     2                       33                       17013
 2025     3                      274                       16773
 2025     4                       22                       17177
 2025     5                        1                       10546


In [10]:
# Scatter comparativo: umidade antes da abertura × volume do evento.
dados_scatter = tabela_linha.dropna(
    subset=['soil_humidity_pre_abertura', 'volume_evento']
).copy()
cores_safra = {'2024': '#2171b5', '2025': '#d73027'}

fig, ax = plt.subplots(figsize=(9, 6))
for safra, dados_safra in dados_scatter.groupby('safra', sort=True):
    linhas = ', '.join(
        str(int(linha)) for linha in sorted(dados_safra['line'].dropna().unique())
    )
    ax.scatter(
        dados_safra['soil_humidity_pre_abertura'],
        dados_safra['volume_evento'],
        color=cores_safra.get(safra, '#636363'),
        edgecolors='#252525', linewidths=0.4, alpha=0.8,
        label=f'Safra {safra} — linha(s) {linhas}',
    )

ax.set_title('Umidade e volume de irrigação por evento de acionamento')
ax.set_xlabel('Umidade do solo antes da abertura (%RH)')
ax.set_ylabel('Volume de água no evento (m³)')
ax.legend(frameon=False)
ax.grid(color='#c6dbef', alpha=0.45)
fig.tight_layout()
saida = ROOT / 'imgs' / 'H1' / 'H1_eventos_umidade_vs_volume_por_safra.png'
saida.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(saida, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Figura salva em: {saida}')

Figura salva em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H1/H1_eventos_umidade_vs_volume_por_safra.png
